# Agentic Automated Fact-Checking Demo

This notebook demonstrates a **real** agentic fact-checking pipeline using **real APIs**:

- Claim → Claim Understanding → Retrieval ↔ Evaluation Loop → Verdict → Explanation
- Inspired by Anthropic's "Building Effective Agents" patterns.

**Requirements:**
- `OPENAI_API_KEY` or `ANTHROPIC_API_KEY` (required)
- `GOOGLE_SEARCH_API_KEY` and `GOOGLE_SEARCH_ENGINE_ID` (optional, for web search)

**Note:** This uses real APIs - no dummies or static data!


## Setup

Install (editable) and ensure the project is on the Python path. If you've already installed locally, you can skip the install cell.


In [ ]:
%pip install pydantic==2.7.0 python-dotenv

In [ ]:
pip install -e 

In [ ]:
# Ensure src/ is on the path for this notebook environment
import sys
import pathlib

ROOT = pathlib.Path("..").resolve()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print("Using project root:", ROOT)


In [ ]:
from factcheck_agent.llm_client import get_default_llm_client
from factcheck_agent.pipeline import FactCheckingPipeline
from factcheck_agent.agents.retrieval import RetrievalAgent
from factcheck_agent.models import Claim
from factcheck_agent.config import get_config

# Get real LLM (will fail if no API key - no dummies!)
try:
    llm = get_default_llm_client(use_dummy_if_missing_key=False)
    print(f"✅ Using LLM: {type(llm).__name__}")
except RuntimeError as e:
    print(f"❌ Error: {e}")
    print("\nPlease set OPENAI_API_KEY or ANTHROPIC_API_KEY in your environment or .env file.")
    raise

# Check Google Search configuration
config = get_config()
has_google_search = bool(
    getattr(config, "GOOGLE_SEARCH_API_KEY", None) 
    and getattr(config, "GOOGLE_SEARCH_ENGINE_ID", None)
)

if has_google_search:
    # Use real Google Search retrieval
    retrieval_agent = RetrievalAgent(llm=llm)
    print("✅ Using Google Search for evidence retrieval")
else:
    print("⚠️  Google Search not configured - using LLM-only retrieval")
    print("   Set GOOGLE_SEARCH_API_KEY and GOOGLE_SEARCH_ENGINE_ID for web search")
    # Create a retrieval agent without Google Search (will use LLM to generate queries but won't search)
    retrieval_agent = RetrievalAgent(llm=llm)

# Create pipeline with real APIs
pipeline = FactCheckingPipeline(llm=llm, retrieval_agent=retrieval_agent)
print("\n🚀 Pipeline ready with real APIs!")
print("   - LLM: Real API")
print("   - Retrieval: Google Search" if has_google_search else "   - Retrieval: LLM-only (no web search)")


## Run example claims

You can either:
1. Use the full pipeline (recommended) - runs everything automatically
2. Run step-by-step to see intermediate results
3. **NEW:** Detect claims from long text and fact-check each one (see cells below)

**Note:** This notebook now uses **real APIs** - no dummies or static data!
- Real LLM for reasoning and explanations
- Real Google Search for evidence retrieval (if configured)
- Real claim detection from articles/posts


In [22]:
# Option 1: Use the full pipeline (simplest - recommended)
# Updated to accept single claim (str or Claim) or multiple claims (list)
async def fact_check_full(claims):
    """
    Run the full pipeline and display results.
    
    Args:
        claims: Can be:
            - A single string (claim text)
            - A single Claim object
            - A single DetectedClaim object
            - A list of any of the above
    
    Returns:
        List of FactCheckResult objects (or single result if single claim provided)
    """
    from factcheck_agent.models import DetectedClaim
    
    # Normalize input to a list
    if isinstance(claims, str):
        # Single string - create a Claim
        claim_list = [Claim(id="claim-0", raw_text=claims)]
        single_result = True
    elif isinstance(claims, Claim):
        claim_list = [claims]
        single_result = True
    elif isinstance(claims, DetectedClaim):
        # Convert DetectedClaim to Claim
        claim_list = [Claim(
            id=claims.id,
            raw_text=claims.raw_text,
            metadata={
                **(claims.metadata or {}),
                "importance": str(claims.importance) if claims.importance is not None else None,
                "sentence_index": str(claims.sentence_index) if claims.sentence_index is not None else None,
            }
        )]
        single_result = True
    elif isinstance(claims, list):
        # List of claims - convert each to Claim
        claim_list = []
        for i, item in enumerate(claims):
            if isinstance(item, str):
                claim_list.append(Claim(id=f"claim-{i}", raw_text=item))
            elif isinstance(item, Claim):
                claim_list.append(item)
            elif isinstance(item, DetectedClaim):
                claim_list.append(Claim(
                    id=item.id,
                    raw_text=item.raw_text,
                    metadata={
                        **(item.metadata or {}),
                        "importance": str(item.importance) if item.importance is not None else None,
                        "sentence_index": str(item.sentence_index) if item.sentence_index is not None else None,
                    }
                ))
        single_result = False
    else:
        raise ValueError(f"Unsupported claim type: {type(claims)}")
    
    results = []
    
    for i, claim in enumerate(claim_list, 1):
        if len(claim_list) > 1:
            print("=" * 80)
            print(f"🔍 Fact-checking claim {i}/{len(claim_list)}: {claim.raw_text[:80]}...")
            print("=" * 80)
        else:
            print("=" * 80)
            print(f"🔍 Fact-checking: {claim.raw_text}")
            print("=" * 80)
        
        print("\n⏳ Processing... This may take 30-60 seconds...\n")
        claim = await pipeline.claim_understanding.normalize_claim_llm(claim)
        
        # # Use LLM normalization if available
        # if pipeline.claim_understanding.llm:
        #     try:
        #         claim = await pipeline.claim_understanding.normalize_claim_llm(claim)
        #     except Exception:
        #         # Fallback to basic normalization
        #         claim = pipeline.claim_understanding.normalize_claim(claim)
        # else:
        #     claim = pipeline.claim_understanding.normalize_claim(claim)
        
        result = await pipeline.process(claim)
        results.append(result)
        
        print("=" * 80)
        print("✅ RESULTS")
        print(f"\n📊 Claim: {claim}" )
        print("=" * 80)
        print(f"\n📊 Verdict: {result.verdict.label}")
        print(f"🎯 Confidence: {result.verdict.confidence:.0%}")
        
        if result.verdict.reasoning:
            print(f"\n🧠 Reasoning: {result.verdict.reasoning}")
        
        print(f"\n💭 Explanation:\n{result.explanation}\n")
        
        if result.evidence:
            print(f"📚 Evidence ({len(result.evidence)} snippet(s)):")
            print("-" * 80)
            for j, e in enumerate(result.evidence[:5], 1):  # Show first 5
                title = e.metadata.get('title', 'No title') if e.metadata else 'No title'
                url = e.metadata.get('url', '') if e.metadata else ''
                print(f"\n[{j}] {title}")
                if url:
                    print(f"    🔗 {url}")
                print(f"    source: {e.source}")
                preview = e.text[:200] + "..." if len(e.text) > 200 else e.text
                print(f"    📄 {preview}")
        else:
            print("📚 No evidence retrieved")
        
        print("\n" + "=" * 80 + "\n")
    
    # Return single result if single claim was provided, otherwise return list
    return results[0] if single_result else results


# Example: Fact-check a single claim
# await fact_check_full("Saudi Arabia is the largest oil producer in the world.")


# Option 2: Step-by-step (see intermediate results)
async def run_demo_step_by_step(text: str):
    """Run step-by-step to see intermediate results."""
    claim = Claim(id="demo-" + text[:8].replace(" ", "_"), raw_text=text)

    # Stage 1: Normalize
    normalized = pipeline.claim_understanding.normalize_claim(claim)
    print("📝 Normalized claim:", normalized.normalized_text)

    # Stage 2: Retrieval (async)
    print("\n🔍 Retrieving evidence...")
    candidates = await pipeline.retrieval.retrieve_evidence(normalized)
    print(f"   Found {len(candidates)} candidate(s):")
    for e in candidates[:3]:  # Show first 3
        preview = e.text[:100] + ("..." if len(e.text) > 100 else "")
        title = e.metadata.get('title', '') if e.metadata else ''
        print(f"   - [{e.source}] {title}")
        print(f"     {preview}")

    # Selection
    selected = pipeline.evidence_selection.select(normalized, candidates, k=5)

    # Evaluation (async)
    print("\n📊 Evaluating evidence...")
    evaluation = await pipeline.evidence_evaluation.evaluate(normalized, selected)
    print(f"   Sufficient: {evaluation.sufficient}")
    print(f"   Confidence: {evaluation.confidence:.0%}")
    if evaluation.notes:
        print(f"   Notes: {evaluation.notes}")

    # Reasoning & verdict (async)
    print("\n🤔 Generating verdict...")
    verdict = await pipeline.reasoning_and_verdict.decide(normalized, evaluation.selected_evidence)
    explanation = await pipeline.explanation.generate(normalized, verdict, evaluation.selected_evidence)

    print(f"\n✅ Verdict: {verdict.label}")
    print(f"🎯 Confidence: {verdict.confidence:.0%}")
    print(f"\n💭 Explanation:\n{explanation}\n")

article = """
In 2020, Saudi Arabia officially launched its National Strategy for Data and Artificial Intelligence (NSDAI) as part of Vision 2030. The strategy aims to position the Kingdom as a global leader in artificial intelligence by 2030, focusing on innovation, talent development, and the use of AI to improve quality of life and government efficiency.

The plan includes training more than 20,000 AI and data specialists, attracting leading global companies, and establishing partnerships with international research institutions. Saudi Arabia has already invested in several AI-driven projects, such as NEOM, The Line city, and smart government initiatives that use data analytics to optimize urban planning and resource management.

The Saudi Data and Artificial Intelligence Authority (SDAIA) oversees the implementation of this strategy, working closely with the Ministry of Communications and Information Technology (MCIT) to ensure the ethical and secure use of AI technologies. The government also hosted the Global AI Summit in Riyadh, bringing together experts from around the world to discuss the future of artificial intelligence and the role of emerging technologies in sustainable development.

Recently, several media outlets reported that Saudi Arabia announced plans to ban the use of foreign AI tools by 2025 to promote locally developed systems. However, no official Saudi source has confirmed such an announcement, and the claim appears to be false or misleading, as the country continues to collaborate internationally in the field of artificial intelligence.
"""


# Example usage (commented out):
# await fact_check_full("Saudi Arabia is the largest oil producer in the world.")
# await fact_check_full(["Claim 1", "Claim 2"])  # Multiple claims


## NEW: Claim Detection from Long Text

**Workflow:**
1. **Provide your article** (cell below)
2. **Detect claims** from the article
3. **Fact-check all detected claims** automatically

This workflow is perfect for analyzing news articles, social media posts, or any long text!


In [27]:
### Step 1: Provide your article

# Edit the `article` variable below with your text, or paste an article from a news source, social media post, etc.


In [24]:
# Paste or type your article here
article = """
Saudi Arabia announced plans to invest $100 billion in renewable energy by 2030, 
according to a statement from the Ministry of Energy. The country, which is currently 
the world's largest oil producer, aims to diversify its energy sources and reduce 
carbon emissions. Climate experts have praised the initiative, noting that it represents 
a significant shift in the country's energy policy. The investment will focus on solar 
and wind power projects across the kingdom. Some analysts believe this move could 
transform the global energy market. The announcement came during a climate summit 
in Riyadh, where officials also revealed plans to achieve net-zero emissions by 2060.
"""

print("📄 Your Article:")
print("=" * 80)
print(article.strip())
print("=" * 80)
print(f"\n📊 Article length: {len(article)} characters")


📄 Your Article:
Saudi Arabia announced plans to invest $100 billion in renewable energy by 2030, 
according to a statement from the Ministry of Energy. The country, which is currently 
the world's largest oil producer, aims to diversify its energy sources and reduce 
carbon emissions. Climate experts have praised the initiative, noting that it represents 
a significant shift in the country's energy policy. The investment will focus on solar 
and wind power projects across the kingdom. Some analysts believe this move could 
transform the global energy market. The announcement came during a climate summit 
in Riyadh, where officials also revealed plans to achieve net-zero emissions by 2060.

📊 Article length: 683 characters


### Step 2: Detect check-worthy claims


In [25]:
from factcheck_agent.agents.claim_understanding import ClaimUnderstandingAgent

# Create claim understanding agent (uses the same LLM as pipeline)
claim_agent = ClaimUnderstandingAgent(llm=llm)

print("🔍 Detecting check-worthy claims from your article...\n")

# Detect claims from the text
# min_importance: only include claims with importance >= 0.5 (adjust as needed)
detected_claims = await claim_agent.detect_claims(article, min_importance=0.5)

print(f"✅ Detected {len(detected_claims)} check-worthy claim(s):\n")
for i, claim in enumerate(detected_claims, 1):
    print(f"[{i}] {claim.raw_text}")
    print(f"    📊 Importance: {claim.importance:.0%}")
    if claim.sentence_index is not None:
        print(f"    📍 Sentence: {claim.sentence_index}")
    print()

if not detected_claims:
    print("⚠️  No claims detected. Try:")
    print("   - Lowering min_importance (e.g., 0.3)")
    print("   - Checking if the text contains factual claims")
    print("   - Making sure the text is long enough")


🔍 Detecting check-worthy claims from your article...

✅ Detected 4 check-worthy claim(s):

[1] Saudi Arabia announced plans to invest $100 billion in renewable energy by 2030, according to a statement from the Ministry of Energy.
    📊 Importance: 100%
    📍 Sentence: 0

[2] The announcement came during a climate summit in Riyadh, where officials also revealed plans to achieve net-zero emissions by 2060.
    📊 Importance: 100%
    📍 Sentence: 6

[3] The country, which is currently the world's largest oil producer, aims to diversify its energy sources and reduce carbon emissions.
    📊 Importance: 90%
    📍 Sentence: 1

[4] The investment will focus on solar and wind power projects across the kingdom.
    📊 Importance: 80%
    📍 Sentence: 4



### Step 3: Fact-check all detected claims

The `fact_check_full()` function will automatically fact-check all detected claims!


In [26]:
# Fact-check all detected claims automatically!
if detected_claims:
    print("🚀 Starting fact-checking for all detected claims...\n")
    results = await fact_check_full(detected_claims)
    
    # Summary
    print("\n" + "=" * 80)
    print("📊 SUMMARY")
    print("=" * 80)
    print(f"\nTotal claims detected: {len(detected_claims)}")
    print(f"Successfully fact-checked: {len(results) if isinstance(results, list) else 1}\n")
    
    print("Results by importance:")
    if isinstance(results, list):
        for dc, result in zip(detected_claims, results):
            verdict_emoji = "✅" if result.verdict.label == "SUPPORTS" else "❌" if result.verdict.label == "REFUTES" else "⚠️"
            print(f"  {verdict_emoji} [{dc.importance:.0%}] {dc.raw_text[:60]}... → {result.verdict.label} ({result.verdict.confidence:.0%})")
    else:
        dc = detected_claims[0]
        verdict_emoji = "✅" if results.verdict.label == "SUPPORTS" else "❌" if results.verdict.label == "REFUTES" else "⚠️"
        print(f"  {verdict_emoji} [{dc.importance:.0%}] {dc.raw_text[:60]}... → {results.verdict.label} ({results.verdict.confidence:.0%})")
else:
    print("⚠️  No claims to fact-check. Run the detection cell above first.")


🚀 Starting fact-checking for all detected claims...

🔍 Fact-checking claim 1/4: Saudi Arabia announced plans to invest $100 billion in renewable energy by 2030,...

⏳ Processing... This may take 30-60 seconds...

✅ RESULTS

📊 Claim: saudi arabia announced plans to invest $100 billion in renewable energy by 2030, according to a statement from the ministry of energy.

📊 Verdict: NOT_ENOUGH_INFO
🎯 Confidence: 80%

🧠 Reasoning: The evidence mentions Saudi Arabia's commitment to renewable energy and Vision 2030 goals, including increasing renewable capacity and investments by various entities, but none explicitly states that Saudi Arabia announced plans to invest $100 billion in renewable energy by 2030 according to a statement from the Ministry of Energy. Therefore, there is insufficient direct evidence to confirm or refute the claim.

💭 Explanation:
The verdict is NOT_ENOUGH_INFO. While the evidence shows that Saudi Arabia is committed to expanding renewable energy capacity and has variou

RetrievalError: Google Search API error: 429 - {
  "error": {
    "code": 429,
    "message": "Quota exceeded for quota metric 'Queries' and limit 'Queries per day' of service 'customsearch.googleapis.com' for consumer 'project_number:65403223194'.",
    "errors": [
      {
        "message": "Quota exceeded for quota metric 'Queries' and limit 'Queries per day' of service 'customsearch.googleapis.com' for consumer 'project_number:65403223194'.",
        "domain": "global",
        "reason": "rateLimitExceeded"
      }
    ],
    "status": "RESOURCE_EXHAUSTED",
    "details": [
      {
        "@type": "type.googleapis.com/google.rpc.ErrorInfo",
        "reason": "RATE_LIMIT_EXCEEDED",
        "domain": "googleapis.com",
        "metadata": {
          "quota_location": "global",
          "quota_unit": "1/d/{project}",
          "consumer": "projects/65403223194",
          "quota_limit_value": "100",
          "quota_limit": "DefaultPerDayPerProject",
          "service": "customsear

## Quick Examples

You can also use `fact_check_full()` with:
- **Single claim string**: `await fact_check_full("Saudi Arabia is the largest oil producer")`
- **Single Claim object**: `await fact_check_full(Claim(id="c1", raw_text="..."))`
- **Single DetectedClaim**: `await fact_check_full(detected_claims[0])`
- **Multiple claims**: `await fact_check_full(["Claim 1", "Claim 2", "Claim 3"])`
- **List of DetectedClaims**: `await fact_check_full(detected_claims)`


In [19]:
# Fact-check all detected claims automatically!
print("🚀 Single Claim")
results = await fact_check_full("Green tea cures COVID-19 within 48 hours")
    
    # Summary
print("\n" + "=" * 80)
print("📊 SUMMARY")
print("=" * 80)
print(f"Successfully fact-checked: {len(results) if isinstance(results, list) else 1}\n")

🚀 Single Claim
🔍 Fact-checking: Green tea cures COVID-19 within 48 hours

⏳ Processing... This may take 30-60 seconds...

✅ RESULTS

📊 Claim: green tea cures covid-19 within 48 hours

📊 Verdict: REFUTES
🎯 Confidence: 90%

🧠 Reasoning: The evidence indicates that claims about green tea curing COVID-19 are false or misleading. Multiple sources mention fake or unproven products claiming to cure COVID-19, including green tea. One source explicitly states that people wrongly believe products like green tea to be a cure. Another source clarifies that green tea does not cure COVID-19 illness. Therefore, the claim that green tea cures COVID-19 within 48 hours is clearly refuted.

💭 Explanation:
The verdict clearly refutes the claim that green tea cures COVID-19 within 48 hours. Multiple reliable sources confirm that green tea does not cure or prevent COVID-19, and claims suggesting otherwise are false or misleading. While some studies have explored antiviral properties of compounds in green te

In [20]:
# Fact-check all detected claims automatically!
print("🚀 Single Claim")
results = await fact_check_full("China will ban TikTok in 2026")
    
    # Summary
print("\n" + "=" * 80)
print("📊 SUMMARY")
print("=" * 80)
print(f"Successfully fact-checked: {len(results) if isinstance(results, list) else 1}\n")

🚀 Single Claim
🔍 Fact-checking: China will ban TikTok in 2026

⏳ Processing... This may take 30-60 seconds...

✅ RESULTS

📊 Claim: china will ban tiktok in 2026

📊 Verdict: NOT_ENOUGH_INFO
🎯 Confidence: 90%

🧠 Reasoning: The evidence discusses various bans and legal actions related to TikTok, primarily focusing on the US banning TikTok or actions taken by the US government and ByteDance's responses. There is no clear or direct information indicating that China will ban TikTok in 2026. The evidence mentions China's ByteDance forming joint ventures and possible easing of China's stance but does not confirm a ban by China in 2026. Therefore, there is insufficient information to confirm or refute the claim.

💭 Explanation:
The verdict is NOT_ENOUGH_INFO. The available evidence primarily discusses actions taken by the US government regarding TikTok, including bans and legal measures, as well as ByteDance's efforts to navigate these restrictions. There is no clear or direct information indic

In [28]:
# Fact-check all detected claims automatically!
print("🚀 Single Claim")
results = await fact_check_full("Saudi Arabia is currently the world's largest oil producer")
    
    # Summary
print("\n" + "=" * 80)
print("📊 SUMMARY")
print("=" * 80)
print(f"Successfully fact-checked: {len(results) if isinstance(results, list) else 1}\n")

🚀 Single Claim
🔍 Fact-checking: Saudi Arabia is currently the world's largest oil producer

⏳ Processing... This may take 30-60 seconds...



RetrievalError: Google Search API error: 429 - {
  "error": {
    "code": 429,
    "message": "Quota exceeded for quota metric 'Queries' and limit 'Queries per day' of service 'customsearch.googleapis.com' for consumer 'project_number:65403223194'.",
    "errors": [
      {
        "message": "Quota exceeded for quota metric 'Queries' and limit 'Queries per day' of service 'customsearch.googleapis.com' for consumer 'project_number:65403223194'.",
        "domain": "global",
        "reason": "rateLimitExceeded"
      }
    ],
    "status": "RESOURCE_EXHAUSTED",
    "details": [
      {
        "@type": "type.googleapis.com/google.rpc.ErrorInfo",
        "reason": "RATE_LIMIT_EXCEEDED",
        "domain": "googleapis.com",
        "metadata": {
          "quota_limit_value": "100",
          "quota_unit": "1/d/{project}",
          "quota_location": "global",
          "quota_metric": "customsearch.googleapis.com/requests",
          "quota_limit": "DefaultPerDayPerProject",
          "s